In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, trim, when, coalesce, to_date, date_format, initcap, sum, avg, count , year, month ,expr

In [0]:
#Setting configuration for storage account
spark.conf.set(
    "fs.azure.account.key.healthstoresa.dfs.core.windows.net",
    "jcqQQwiqx1bSrdThPx+yCNBKabOFSAjIx7YY8ZeCJO8O8B4WgpQH48wTiKqMvWcHQyjyjl+jA0A1+AStvV5enA=="
)


In [0]:
#Getting new file name using trigger
dbutils.widgets.text("inputFileProp", "")
file_path = dbutils.widgets.get("inputFileProp")
print(f"Automatically triggered for file: {file_path}")

In [0]:
#Loading raw csv file and then saving as delta file in bronze layer
rawSource = f"abfss://inputdata@healthstoresa.dfs.core.windows.net/{file_path}"
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .load(rawSource)
#Saving bronze layer
bronzePath = "abfss://outputdata@healthstoresa.dfs.core.windows.net/bronze/events_delta"

if DeltaTable.isDeltaTable(spark, bronzePath):
    # If it exists, append or merge depending on your duplicate policy.
    # To simply append safely:
    df.write.mode("append").format("delta").save(bronzePath)
else:
    # If it's the very first run, use overwrite to cleanly initialize it 
    # without concurrent protocol clashes
    df.write.mode("overwrite").format("delta").save(bronzePath)
bronzeDF=df


In [0]:
#Now let's first create multiple tables from this dataset
user_activity_dfb = bronzeDF.select(
    "user_id",
    "date",
    "steps",
    "distance_km",
    "active_minutes"
)
workout_dfb = bronzeDF.select(
    "user_id",
    "date",
    "workout_type",
    "calories_burned",
    "heart_rate_avg"
)
sleep_dfb = bronzeDF.select(
    "user_id",
    "date",
    "sleep_hours",
    "mood"
)
location_dfb=bronzeDF.select(
    "user_id",
    "weather_conditions",
    "location"
)
mood_dfb=bronzeDF.select(
    "user_id",
    "date",
    "mood"
)


Now we will store these tables as delta files

In [0]:
userActivityPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/bronze/userActivityDelta"
workoutPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/bronze/workoutDelta"
sleepPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/bronze/sleepDelta"
locationPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/bronze/locationDelta"
moodPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/bronze/moodDelta"

Now writing them respectively

In [0]:
user_activity_dfb.write.mode("append").format("delta").save(userActivityPath)
workout_dfb.write.mode("append").format("delta").save(workoutPath)
sleep_dfb.write.mode("append").format("delta").save(sleepPath)
location_dfb.write.mode("append").format("delta").save(locationPath)
mood_dfb.write.mode("append").format("delta").save(moodPath)

Now for performing silver layer we will use bronze dataframes then process them and finally save them in silver datafram

In [0]:
#First for user activity
#Handling NULL and empty strings in user_id
user_activity_dfb = user_activity_dfb.replace("NULL", None)

user_activity_dfb = user_activity_dfb.withColumn(
    "user_id",
    when(trim(col("user_id")) == "", None)
    .otherwise(col("user_id"))
)
#Handling mixed date format
user_activity_dfb = user_activity_dfb.withColumn(
    "parsed_date",
    coalesce(
        expr("try_to_date(date, 'MMMM dd, yyyy')"),
        expr("try_to_date(date, 'dd-MM-yy')")
    )
)
user_activity_dfb = user_activity_dfb.withColumn(
    "date",
    date_format(col("parsed_date"), "dd-MM-yyyy")
).drop("parsed_date")

#Converting numeric columns to actual numeric data
user_activity_dfb = user_activity_dfb.withColumn(
    "user_id",
    col("user_id").cast("int")
)

user_activity_dfb = user_activity_dfb.withColumn(
    "steps",
    col("steps").cast("int")
)

user_activity_dfb = user_activity_dfb.withColumn(
    "distance_km",
    col("distance_km").cast("double")
)

user_activity_dfb = user_activity_dfb.withColumn(
    "active_minutes",
    col("active_minutes").cast("int")
)

user_activity_dfb = user_activity_dfb.dropDuplicates(
    ["user_id", "date"]
)


In [0]:
#Processing workout data
#Replacing null values
workout_dfb = workout_dfb.replace("NULL", None)
#Standardizing the date format

workout_dfb = workout_dfb.withColumn(
    "parsed_date",
    coalesce(
        expr("try_to_date(date, 'MMMM dd, yyyy')"),
        expr("try_to_date(date, 'dd-MM-yy')")
    )
)
workout_dfb = workout_dfb.withColumn(
    "date",
    date_format(col("parsed_date"), "dd-MM-yyyy")
).drop("parsed_date")

#Removing trailing spaces and converting to title case
workout_dfb = workout_dfb.withColumn(
    "workout_type",
    initcap(trim(col("workout_type")))
)
#Converting numeric columns to actual numeric data
workout_dfb = workout_dfb.withColumn(
    "user_id",
    col("user_id").cast("int")
)

workout_dfb = workout_dfb.withColumn(
    "calories_burned",
    col("calories_burned").cast("double")
)

workout_dfb = workout_dfb.withColumn(
    "heart_rate_avg",
    col("heart_rate_avg").cast("int")
)

workout_dfb = workout_dfb.dropDuplicates(
    ["user_id", "date", "workout_type"]
)

In [0]:
#Now processing sleep dataframe
#Replacing null values
sleep_dfb = sleep_dfb.replace("NULL", None)
#Standardizing the date format
sleep_dfb = sleep_dfb.withColumn(
    "parsed_date",
    coalesce(
        expr("try_to_date(date, 'MMMM dd, yyyy')"),
        expr("try_to_date(date, 'dd-MM-yy')")
    )
)
sleep_dfb = sleep_dfb.withColumn(
    "date",
    date_format(col("parsed_date"), "dd-MM-yyyy")
).drop("parsed_date")
#Removing trailing spaces and converting to title case
sleep_dfb = sleep_dfb.withColumn(
    "mood",
    initcap(trim(col("mood")))
)
#Converting numeric columns to actual numeric data
sleep_dfb = sleep_dfb.withColumn(
    "user_id",
    col("user_id").cast("int")
)

sleep_dfb = sleep_dfb.withColumn(
    "sleep_hours",
    col("sleep_hours").cast("double")
)

sleep_dfb = sleep_dfb.dropDuplicates(
    ["user_id", "date"]
)

In [0]:
#Fourth we have location dataframe
#Replacing null values
location_dfb = location_dfb.replace("NULL", None)
#Removing trailing spaces and converting to title case
location_dfb = location_dfb.withColumn(
    "location",
    initcap(trim(col("location")))
)
location_dfb = location_dfb.withColumn(
    "weather_conditions",
    initcap(trim(col("weather_conditions")))
)
#Converting to numeric columns
location_dfb = location_dfb.withColumn(
    "user_id",
    col("user_id").cast("int")
)
#Dropping duplicates using all columns
location_dfb = location_dfb.dropDuplicates(
    ["user_id", "location", "weather_conditions"]
)

In [0]:
#Finally mood dataframe
#Replacing null values
mood_dfb = mood_dfb.replace("NULL", None)
#Standardizing the date format
mood_dfb = mood_dfb.withColumn(
    "parsed_date",
    coalesce(
        expr("try_to_date(date, 'MMMM dd, yyyy')"),
        expr("try_to_date(date, 'dd-MM-yy')")
    )
)
mood_dfb = mood_dfb.withColumn(
    "date",
    date_format(col("parsed_date"), "dd-MM-yyyy")
).drop("parsed_date")
#Removing trailing spaces and converting to title case
mood_dfb = mood_dfb.withColumn(
    "mood",
    initcap(trim(col("mood")))
)
#Converting to numeric columns
mood_dfb = mood_dfb.withColumn(
    "user_id",
    col("user_id").cast("int")
)
#Removing duplicates
mood_dfb = mood_dfb.dropDuplicates(
    ["user_id", "date"]
)

Now we have cleaned data , we will use upsert logic here to merge them with silver delta table

In [0]:
userActivityPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/silver/userActivityDelta"
workoutPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/silver/workoutDelta"
sleepPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/silver/sleepDelta"
locationPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/silver/locationDelta"
moodPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/silver/moodDelta"

In [0]:
#Upsert for userActivity delta files

if DeltaTable.isDeltaTable(spark, userActivityPath):
    deltaTable = DeltaTable.forPath(spark, userActivityPath)
    
    deltaTable.alias("target").merge(
        user_activity_dfb.alias("source"),
        "target.user_id = source.user_id AND target.date = source.date"
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()
    print("Upsert completed successfully.")

else:
    user_activity_dfb.write \
        .format("delta") \
        .mode("overwrite") \
        .save(userActivityPath)
    print("Initial Delta Table created.")


Upsert completed successfully.


In [0]:
#Upsert for workout delta files

if DeltaTable.isDeltaTable(spark, workoutPath):
    deltaTable = DeltaTable.forPath(spark, workoutPath)
    
    deltaTable.alias("target").merge(
        workout_dfb.alias("source"),
        "target.user_id = source.user_id AND target.date = source.date AND target.workout_type = source.workout_type"
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()
    print("Upsert completed successfully.")

else:
    workout_dfb.write \
        .format("delta") \
        .mode("overwrite") \
        .save(workoutPath)
    print("Initial Delta Table created.")

Upsert completed successfully.


In [0]:
#Upsert for sleepData delta files

if DeltaTable.isDeltaTable(spark, sleepPath):
    deltaTable = DeltaTable.forPath(spark, sleepPath)
    
    deltaTable.alias("target").merge(
        sleep_dfb.alias("source"),
        "target.user_id = source.user_id AND target.date = source.date"
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()
    print("Upsert completed successfully.")

else:
    sleep_dfb.write \
        .format("delta") \
        .mode("overwrite") \
        .save(sleepPath)
    print("Initial Delta Table created.")

Upsert completed successfully.


In [0]:
#Upsert for locationBased delta files

if DeltaTable.isDeltaTable(spark, locationPath):
    deltaTable = DeltaTable.forPath(spark, locationPath)
    
    deltaTable.alias("target").merge(
        location_dfb.alias("source"),
        "target.user_id = source.user_id AND target.location = source.location"
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()
    print("Upsert completed successfully.")

else:
    location_dfb.write \
        .format("delta") \
        .mode("overwrite") \
        .save(locationPath)
    print("Initial Delta Table created.")

Upsert completed successfully.


In [0]:
#Upsert for mood data delta files

if DeltaTable.isDeltaTable(spark, moodPath):
    deltaTable = DeltaTable.forPath(spark, moodPath)
    
    deltaTable.alias("target").merge(
        mood_dfb.alias("source"),
        "target.user_id = source.user_id AND target.date = source.date"
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()
    print("Upsert completed successfully.")

else:
    mood_dfb.write \
        .format("delta") \
        .mode("overwrite") \
        .save(moodPath)
    print("Initial Delta Table created.")

Upsert completed successfully.


Now we are in final gold layer 

In [0]:
userActivityPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/gold/userActivityDelta"
workoutPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/gold/workoutDelta"
sleepPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/gold/sleepDelta"
locationPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/gold/locationDelta"
moodPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/gold/moodDelta"


In [0]:
#Creating gold table for user activity
gold_user_summary = user_activity_dfb.groupBy(
    "user_id"
).agg(
    sum("steps").alias("total_steps"),
    avg("steps").alias("avg_steps"),
    sum("distance_km").alias("total_distance"),
    avg("active_minutes").alias("avg_active_minutes")
)
gold_user_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .save(userActivityPath)

In [0]:
#Creating gold table for workout data
gold_workout_performance = workout_dfb.groupBy(
    "workout_type"
).agg(
    avg("calories_burned").alias("avg_calories"),
    avg("heart_rate_avg").alias("avg_heart_rate"),
    count("*").alias("total_sessions")
)
gold_workout_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .save(workoutPath)

In [0]:
#Creating gold table for sleep data
gold_sleep_analysis = sleep_dfb.groupBy(
    "mood"
).agg(
    avg("sleep_hours").alias("avg_sleep_hours"),
    count("*").alias("record_count")
)
gold_sleep_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .save(sleepPath)

In [0]:
#Creating gold table for location data
gold_location_analysis = location_dfb.groupBy(
    "location"
).agg(
    count("*").alias("visit_count")
)
gold_location_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .save(locationPath)

In [0]:
#Creating gold table for mood data
gold_mood_analysis = mood_dfb.groupBy(
    "mood"
).agg(
    count("*").alias("total_occurrences")
)
gold_mood_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .save(moodPath)

In [0]:
#Creating gold table for month and year
gold_monthly_activity = user_activity_dfb.groupBy(
    "date"
).agg(
    sum("steps").alias("total_steps")
)
#This will not be stored in the delta table , we are going to directyl use it in power bi dashboard only

In [0]:

userActivityPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/gold/userActivityDelta"
workoutPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/gold/workoutDelta"
sleepPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/gold/sleepDelta"
locationPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/gold/locationDelta"
moodPath="abfss://outputdata@healthstoresa.dfs.core.windows.net/gold/moodDelta"
spark.sql(f"CREATE DATABASE IF NOT EXISTS gold;")
#We will use these tables from SQL Warehouse in our power bi
gold_user_summary.write.mode("overwrite").saveAsTable("gold.user_summary")
gold_workout_performance.write.mode("overwrite").saveAsTable("gold.workout_performance")
gold_sleep_analysis.write.mode("overwrite").saveAsTable("gold.sleep_analysis")
gold_location_analysis.write.mode("overwrite").saveAsTable("gold.location_data")
gold_mood_analysis.write.mode("overwrite").saveAsTable("gold.mood_analysis")
gold_monthly_activity.write.mode("overwrite").saveAsTable("gold.monthly_activity")
